PRODUCER FOR NEWS API 

In [27]:
import os
import json 
from datetime import datetime, timedelta 
from kafka import KafkaProducer 
from dotenv import load_dotenv
from newsapi import NewsApiClient

In [28]:
load_dotenv() 

NEWS_API_KEY = os.getenv('NEWS_API_KEY')
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC")

In [29]:
producer = KafkaProducer( 
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    key_serializer=lambda k : k.encode("utf-8") if k else None,
)

In [30]:
newsAPI = NewsApiClient(api_key=NEWS_API_KEY)

producerNews = producer

from_date = (datetime.now() - timedelta(days=3)).strftime("%Y-%m-%d")

print(f"[{datetime.now()}] FETCHING TOP HEADLINES FROM NEWSAPI...")

try : 
    response = newsAPI.get_everything(
        q='china',
        language='en',
        from_param=from_date,
        sort_by='relevancy',
        page_size=20
        )

    articles = response.get("articles", [])

    print(f"FETCHED {len(articles)}")

    for article in articles :
        record = { 
            "source": article.get("source", {}).get("name"),
            "author": article.get("author"),
            "title": article.get("title"),
            "description": article.get("description"),
            "url": article.get("url"),
            "published_at": article.get("publishedAt"),
            "content": article.get("content"),
            "fetched_at": datetime.now().isoformat()
        }

        key = article.get("url") or article.get("title")
        producerNews.send(KAFKA_TOPIC, value=record, key=key)
        print(f" SENT {record['title'][:80]} ... ")

    producerNews.flush()
    print(f"[{datetime.now()}] SUCCESS sent {len(articles)} articles to topic {KAFKA_TOPIC}")

except Exception as e :
    print(f"[{datetime.now()}] ERROR: {e}")
finally:
    producerNews.close()

[2026-09-10 12:40:40.353168] FETCHING TOP HEADLINES FROM NEWSAPI...
FETCHED 19
 SENT Silicon Valley’s AI Agent Push Has Been Paying Off—for Cybercriminals ... 
 SENT Huawei copies Samsung’s privacy display in its latest trifold ... 
 SENT Foldable 'iPhone Ultra' May Launch Later in China, Says Leaker ... 
 SENT Rapist serving life may face more assault charges ... 
 SENT ARM’s Next-Gen Phone Chips Are So Much Better for Gaming. Does it Matter? ... 
 SENT I'm an airline employee who flies for free, sometimes even in business class. It ... 
 SENT As Nepal identifies those killed in the floods, questions grow over China's repo ... 
 SENT The US plans to raise AI-directed cyberattacks with China, Nikkei reports ... 
 SENT EU trade chief will ask CEOs how much decoupling from China they can afford ... 
 SENT China put a 99.2% duty on a chemical chip fabs need ... 
 SENT China has 70% of a material AI needs, IQE’s CEO warns, Bloomberg reports ... 
 SENT Belgian chip researcher held since May

CONSUMER FOR THE NEWS API 

In [31]:
import os 
import json 
from dotenv import load_dotenv
from kafka import KafkaConsumer 
import pandas as pd 

from pyiceberg.catalog import load_catalog
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, StringType

import pyarrow as pa 

In [32]:
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC")
KAFKA_GROUP_ID = os.getenv("KAFKA_GROUP_ID")

In [33]:
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION")
S3_ENDPOINT = os.getenv("S3_ENDPOINT")
ICEBERG_REST_URI = os.getenv("ICEBERG_REST_URI")
WAREHOUSE = os.getenv("WAREHOUSE")

In [34]:
 catalogNews = load_catalog( 
        "rest",
        type="rest",
        uri=ICEBERG_REST_URI,
        warehouse=WAREHOUSE,
        
        **{ 
            "s3.endpoint": S3_ENDPOINT,
            "s3.access-key-id": AWS_ACCESS_KEY_ID,
            "s3.secret-access-key": AWS_SECRET_ACCESS_KEY,
            "s3.region": AWS_REGION,
            "s3.path-style-access": "true",
        }
    )

In [35]:
def get_or_create_table(catalog): 
    namespace = "news"
    table_name = "news_raw"
    full_name = f"{namespace}.{table_name}"

    try: 
        table = catalogNews.load_table(full_name)
        print(f"Table {full_name} already exists.")
        return table
    
    except Exception:
        print(f"Creating table {full_name}")

        try : 
            catalogNews.create_namespace(namespace)

        except Exception: 
            pass

        schema = Schema( 
            NestedField(1, "source", StringType(), required=False),
            NestedField(2, "author", StringType(), required=False),
            NestedField(3, "title", StringType(), required=False),
            NestedField(4, "description", StringType(), required=False),
            NestedField(5, "url", StringType(), required=False),
            NestedField(6, "published_at", StringType(), required=False),
            NestedField(7, "content", StringType(), required=False),
            NestedField(8, "fetched_at", StringType(), required=False),
        )

        table = catalogNews.create_table( 
            identifier=full_name, 
            schema = schema , 
            location = f"{WAREHOUSE}{namespace}/{table_name}"
        )

        print(f"Table {full_name} created successfully")
        return table

In [36]:
print("Starting News Consumer")
print(f"Listening to Topic {KAFKA_TOPIC}")

catalog = catalogNews
table = get_or_create_table(catalog)

consumer = KafkaConsumer(
    KAFKA_TOPIC,
    bootstrap_servers = KAFKA_BOOTSTRAP_SERVERS,
    group_id = KAFKA_GROUP_ID,
    auto_offset_reset = "earliest",
    enable_auto_commit= True,
    value_deserializer=lambda x: json.loads(x.decode("utf-8"))
)

batch = []

try: 
    for message in consumer: 
        record = message.value
        print(f"Received {record.get('title', '')[:70]}...")

        batch.append(record)

except KeyboardInterrupt:
    print("\nStopping consumer...")

    if batch : 
        df = pd.DataFrame(batch)
        pa_table = pa.Table.from_pandas(df, preserve_index=False)
        table.append(pa_table)
        print(f"Wrote Remaining {len(batch)} record")

finally : 
    consumer.close() 
    print("Consumer stopped")

Starting News Consumer
Listening to Topic news_raw
Table news.news_raw already exists.


/tmp/ipykernel_32999/2776338576.py:7: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Received Silicon Valley’s AI Agent Push Has Been Paying Off—for Cybercriminals...
Received Huawei copies Samsung’s privacy display in its latest trifold...
Received Foldable 'iPhone Ultra' May Launch Later in China, Says Leaker...
Received Rapist serving life may face more assault charges...
Received ARM’s Next-Gen Phone Chips Are So Much Better for Gaming. Does it Matt...
Received I'm an airline employee who flies for free, sometimes even in business...
Received As Nepal identifies those killed in the floods, questions grow over Ch...
Received The US plans to raise AI-directed cyberattacks with China, Nikkei repo...
Received EU trade chief will ask CEOs how much decoupling from China they can a...
Received China put a 99.2% duty on a chemical chip fabs need...
Received China has 70% of a material AI needs, IQE’s CEO warns, Bloomberg repor...
Received Belgian chip researcher held since May over secrets allegedly sent to ...
Received Philippines' Teodoro says China could exploit cuts in

SENTIMENT ANALYST

In [38]:
import os 
import numpy as np 
import pandas as pd 
import pyarrow as pa 
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, StringType, DoubleType, IntegerType

In [39]:
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION")
S3_ENDPOINT = os.getenv("S3_ENDPOINT")
ICEBERG_REST_URI = os.getenv("ICEBERG_REST_URI")
WAREHOUSE = os.getenv("WAREHOUSE")
 
N_CLUSTERS = int(os.getenv("ML_N_CLUSTERS", "8"))
 
RAW_NAMESPACE = "news"
RAW_TABLE = "news_raw"
ENRICHED_NAMESPACE = "news"
ENRICHED_TABLE = "news_sentiment"

In [40]:
def load_raw_article(catalog) -> pd.DataFrame: 
    full_name = f"{RAW_NAMESPACE}.{RAW_TABLE}"
    table = catalog.load_table(full_name)
    df = table.scan().to_pandas()
    print(f"Loaded {len(df)} rows from {full_name}")

    return df

In [41]:
def sentiment_analyst(df: pd.DataFrame, n_clusters: int=N_CLUSTERS) -> pd.DataFrame :
    if df.empty:
        print("No Rows to enrich")
        return df 
    
    df = df.copy()
    df["text_for_analyst"] = (
        df["title"].fillna("") + ". " + df["description"].fillna("")
    )

    analyzer = SentimentIntensityAnalyzer()
    scores = df["text_for_analyst"].apply(analyzer.polarity_scores)
    df["sentiment_score"] = scores.apply(lambda s :s["compound"]).astype(float)

    def sentiment_score(score: float)-> str : 
        if score >= 0.05:
            return "positive"
        elif score <= -0.05:
            return "negative"
        return "neutral"
    
    df["sentiment_label"] = df["sentiment_score"].apply(sentiment_score)

    vectorizer = TfidfVectorizer ( 
        max_features=1000,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=1,
    )

    tfdfi_matrix = vectorizer.fit_transform(df["text_for_analyst"])

    k=min(n_clusters, len(df))
    kmeans =KMeans(n_clusters=k, random_state=42, n_init=10)
    df["topic_cluster"] = kmeans.fit_predict(tfdfi_matrix).astype("int32")

    terms = np.array(vectorizer.get_feature_names_out())
    order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
    cluster_labels = { 
        i: ", ".join(terms[order_centroids[i, :3]]) for i in range(k)
    }

    df["topic_label"] = df["topic_cluster"].map(cluster_labels)
 
    return df.drop(columns=["text_for_analyst"])

In [42]:
def get_or_create_enriched_table(catalog, sample_df: pd.DataFrame):
    full_name = f"{ENRICHED_NAMESPACE}.{ENRICHED_TABLE}"
 
    try:
        table = catalog.load_table(full_name)
        print(f"Table {full_name} already exists.")
        return table
    except Exception:
        print(f"Creating table {full_name}")
 
        try:
            catalog.create_namespace(ENRICHED_NAMESPACE)
        except Exception:
            pass
 
        schema = Schema(
            NestedField(1, "source", StringType(), required=False),
            NestedField(2, "author", StringType(), required=False),
            NestedField(3, "title", StringType(), required=False),
            NestedField(4, "description", StringType(), required=False),
            NestedField(5, "url", StringType(), required=False),
            NestedField(6, "published_at", StringType(), required=False),
            NestedField(7, "content", StringType(), required=False),
            NestedField(8, "fetched_at", StringType(), required=False),
            NestedField(9, "sentiment_score", DoubleType(), required=False),
            NestedField(10, "sentiment_label", StringType(), required=False),
            NestedField(11, "topic_cluster", IntegerType(), required=False),
            NestedField(12, "topic_label", StringType(), required=False),
        )
 
        table = catalog.create_table(
            identifier=full_name,
            schema=schema,
            location=f"{WAREHOUSE}{ENRICHED_NAMESPACE}/{ENRICHED_TABLE}",
        )
        print(f"Table {full_name} created successfully")
        return table

In [43]:
def write_enriched(table, df: pd.DataFrame):
    if df.empty:
        print("Nothing to write.")
        return
 
    # Column order must match the Iceberg schema field order
    ordered_cols = [
        "source", "author", "title", "description", "url",
        "published_at", "content", "fetched_at",
        "sentiment_score", "sentiment_label", "topic_cluster", "topic_label",
    ]
    df = df[ordered_cols]
 
    pa_table = pa.Table.from_pandas(df, preserve_index=False)
 
    # Full overwrite: keeps clustering consistent across runs (see module
    # docstring). Swap to table.append(pa_table) later if you move to a
    # true incremental design.
    table.overwrite(pa_table)
    print(f"Wrote {len(df)} enriched rows.")

In [44]:
def main():
    print("Starting ML enrichment step")
    catalog = catalogNews
 
    raw_df = load_raw_article(catalog)
    enriched_df = sentiment_analyst(raw_df)
 
    enriched_table = get_or_create_enriched_table(catalog, enriched_df)
    write_enriched(enriched_table, enriched_df)
 
    print("ML enrichment complete.")

main()

Starting ML enrichment step
Loaded 38 rows from news.news_raw
Creating table news.news_sentiment
Table news.news_sentiment created successfully
Wrote 38 enriched rows.
ML enrichment complete.


/home/aceino/08.Project/newsAPIProject/.venv/lib/python3.14/site-packages/pyiceberg/table/__init__.py:721: UserWarning: Delete operation did not match any records
  self.delete(


THE CONNECTION FROM ICEBERG TO DUCKDB

In [51]:
import duckdb 

con = duckdb.connect()

con.sql ("INSTALL httpfs; LOAD httpfs")
con.sql ("INSTALL iceberg; LOAD iceberg")

con.sql(
        """
    CREATE secret minio_secret(
        TYPE s3, 
        KEY_ID 'admin', 
        SECRET 'password', 
        ENDPOINT 'localhost:9000', 
        URL_STYLE 'path', 
        USE_SSL false
    );
    """
)

con.sql(
    """
    attach '' as iceberg_catalog( 
        TYPE ICEBERG, 
        ENDPOINT 'http://localhost:8181',
        AUTHORIZATION_TYPE 'none',
        ACCESS_DELEGATION_MODE 'none'
        );
    """
)

print("Attached. Tables visible to   Duckdb")
print(con.sql("Show all tables"))

print("/n Quering news.news_sentiment directly through Duckdb")
result = con.sql("Select * from iceberg_catalog.news.news_sentiment limit 5;")

print(result)

Attached. Tables visible to   Duckdb
┌─────────────────┬─────────┬────────────────┬──────────────┬──────────────┬───────────┐
│    database     │ schema  │      name      │ column_names │ column_types │ temporary │
│     varchar     │ varchar │    varchar     │  varchar[]   │  varchar[]   │  boolean  │
├─────────────────┼─────────┼────────────────┼──────────────┼──────────────┼───────────┤
│ iceberg_catalog │ news    │ news_raw       │ [__]         │ [UNKNOWN]    │ false     │
│ iceberg_catalog │ news    │ news_sentiment │ [__]         │ [UNKNOWN]    │ false     │
└─────────────────┴─────────┴────────────────┴──────────────┴──────────────┴───────────┘

/n Quering news.news_sentiment directly through Duckdb
┌─────────────┬──────────────────────────────────┬───────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────